In [1]:
import numpy as np

In [3]:
vcf_content = """##
#CHROM	POS	ID	REF	ALT	QUAL	FILTER	INFO	FORMAT	x1	x2	x3	y1	y2	a1	a2	a3	a4
1	1	.	A	T	.	.	.	GT	0/0	0/1	0/1	1|1	1|1	0/1	1/1	0/1	0/1
1	11	.	A	T	.	.	.	GT	0/.	0/0	0/0	1/1	1|1	0/0	0/1	0/0	0/1
1	21	.	A	T	.	.	.	GT	1/1	1/1	0/1	0|0	0|0	0/1	0/1	0/1	0/0
1	31	.	A	T	.	.	.	GT	0/1	1/1	1/1	1/1	1/1	0/0	1/1	1/1	1/1
2	1	.	A	T	.	.	.	GT	0/0	0/0	0/1	0/1	0/1	1/1	1/1	1/1	1/1
2	11	.	A	T	.	.	.	GT	.	0/0	0/1	1|1	1|1	0/1	1/1	0/1	0/1
2	21	.	A	T	.	.	.	GT	1/1	0/1	0/1	0|1	1|0	0/1	1/1	0/1	0/1
"""
with open("data/example.vcf", "w") as fout:
    fout.write(vcf_content)

sample_content = """x1 0 
x2 0
x3 0
y1 1
y2 1
a1 admixed
a2 admixed
a3 admixed
a4 admixed
"""
with open("data/example.samples", "w") as fout:
    fout.write(sample_content)

rec_content = """Chrom\tStart\tStop\tcM/Mb
1\t1\t10\t0.01
1\t10\t20\t0.01
1\t20\t50\t0.01
2\t1\t30\t1
"""
with open("data/example.rec.tsv", "w") as fout:
    fout.write(rec_content)

# expected
panels = [
    [4, 2, 0, 4],
    [5, 0, 0, 4],
    [1, 5, 4, 0],
    [1, 5, 0, 4],
    [5, 1, 2, 2],
    [3, 1, 0, 4],
    [2, 4, 2, 2]
]
samples = [
    [1, 1, 0, 2, 1, 1, 1, 1],
    [2, 0, 1, 1, 2, 0, 1, 1],
    [1, 1, 1, 1, 1, 1, 2, 0,],
    [2, 0, 0, 2, 0, 2, 0, 2],
    [0, 2, 0, 2, 0, 2, 0, 2],
    [1, 1, 0, 2, 1, 1, 1, 1],
    [1, 1, 0, 2, 1, 1, 1, 1],
]

In [17]:
!python ../scripts/vcf_to_ahmm.py -v data/example.vcf -s data/example.samples \
    -r data/example.rec.tsv -g 1 -m 1 -o data/example.ahmm.samples > data/example.ahmm.tsv
!cat data/example.ahmm.tsv

1	1	4	2	0	4	0.0	1	1	0	2	1	1	1	1
1	11	5	0	0	4	1e-09	2	0	1	1	2	0	1	1
1	21	1	5	4	0	1e-09	1	1	1	1	1	1	2	0
1	31	1	5	0	4	1.0000000000000003e-09	2	0	0	2	0	2	0	2
2	1	5	1	2	2	0.0	0	2	0	2	0	2	0	2
2	11	3	1	0	4	1e-07	1	1	0	2	1	1	1	1
2	21	2	4	2	2	1e-07	1	1	0	2	1	1	1	1


In [26]:
!rm data/example.ahmm.split.*
!python ../scripts/split_ahmm_input.py -i data/example.ahmm.tsv -s data/example.samples \
    -o data/example.ahmm.split.
!cat data/example.ahmm.split.a1.tsv
!echo ""
!cat data/example.ahmm.split.a2.tsv

1	1	4	2	0	4	0.0	1	1
1	11	5	0	0	4	1e-09	2	0
1	21	1	5	4	0	1e-09	1	1
1	31	1	5	0	4	1.0000000000000003e-09	2	0
2	1	5	1	2	2	0.0	0	2
2	11	3	1	0	4	1e-07	1	1
2	21	2	4	2	2	1e-07	1	1

1	1	4	2	0	4	0.0	0	2
1	11	5	0	0	4	1e-09	1	1
1	21	1	5	4	0	1e-09	1	1
1	31	1	5	0	4	1.0000000000000003e-09	0	2
2	1	5	1	2	2	0.0	0	2
2	11	3	1	0	4	1e-07	0	2
2	21	2	4	2	2	1e-07	0	2


In [6]:
!ancestry_hmm -i data/example.ahmm.split.a1.tsv -s data/example.ahmm.split.a1.samples \
    -a 2 0.8 0.2 -p 0 0.8 1598 -p 1 0.2 1598 \ -ne 2500000 \ -g
!mv a1.posterior data/
!cat data/a1.posterior

reading command line				33 ms
reading sample ids and ploidy			108 ms
creating states matrix				7 ms
reading data and creating emissions matrices	81 ms
computing transition routes			13 ms
creating initial admixture model(s)		1 ms
estimating 0 parameters


evaluating supplied model:

	type	time	proportion
optimum: 
	type	time	proportion
	0	0.8	0.8
	0	0.8	0.8
	1	0.2	0.2
	1	0.2	0.2

creating per morgan transition rates		3 ms
creating transition matrices			7 ms
computing forward probabilities	lnl: -49.1554		8 ms
computing backward probabilities			2 ms
forward-backward posterior decoding and printing			138 ms
total run time:			581 ms
chrom	position	2,0	1,1	0,2
1	1	0.47324	0.515368	0.0113922
1	11	0.74969	0.246324	0.00398635
1	21	0.393955	0.594452	0.0115935
1	31	0.614889	0.36949	0.0156212
2	1	0.113086	0.464519	0.422395
2	11	0.159091	0.673302	0.167607
2	21	0.170323	0.551415	0.278262


In [27]:
!ancestry_hmm -i data/example.ahmm.split.a2.tsv -s data/example.ahmm.split.a2.samples \
    -a 2 0.8 0.2 -p 0 0.8 1598 -p 1 0.2 1598 \ -ne 2500000 \ -g
!mv a2.posterior data/
!cat data/a2.posterior

reading command line				34 ms
reading sample ids and ploidy			97 ms
creating states matrix				7 ms
reading data and creating emissions matrices	77 ms
computing transition routes			13 ms
creating initial admixture model(s)		0 ms
estimating 0 parameters


evaluating supplied model:

	type	time	proportion
optimum: 
	type	time	proportion
	0	0.8	0.8
	0	0.8	0.8
	1	0.2	0.2
	1	0.2	0.2

creating per morgan transition rates		3 ms
creating transition matrices			6 ms
computing forward probabilities	lnl: -45.8897		8 ms
computing backward probabilities			2 ms
forward-backward posterior decoding and printing			125 ms
total run time:			570 ms
chrom	position	2,0	1,1	0,2
1	1	0.038307	0.669824	0.291869
1	11	0.0281737	0.889169	0.0826571
1	21	0.0443309	0.866946	0.0887231
1	31	0.0662045	0.727765	0.20603
2	1	0.0560626	0.191656	0.752281
2	11	0.0424006	0.174592	0.783007
2	21	0.13813	0.264955	0.596915


In [33]:
samples_to_merge = """a1 admixed
a2 admixed
"""
with open("data/samples.to.merge", "w") as fout:
    fout.write(samples_to_merge)

!python ../scripts/merge_ahmm_outputs.py -i data/a1.posterior data/a2.posterior -s data/samples.to.merge \
    -o data/merged.ancestries.tsv
!cat data/merged.ancestries.tsv

chrom	position	a1	a2
1	1	-1	-1
1	11	0	-1
1	21	-1	-1
1	31	0	-1
2	1	-1	1
2	11	-1	1
2	21	-1	1


In [23]:
samples

[]